# FAME — DRVM in the real Cartola and Energy applications (V10)

V10 preserves the same DRVM detector and parameters used in V9. The only substantive correction is input handling for the Cartola D→C→R→E protocol: 2023 calibration/freeze, independent 2024 reference, and 2025 deployment are now read from the same audited Cartola extension root.


In [ ]:

from __future__ import annotations

import math
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
OUT = ROOT / "results_drvm_real_v10"
OUT.mkdir(parents=True, exist_ok=True)

ALPHA = 0.04
GAMMA = 0.01
DELTA_X = 0.0
LAMBDAS = np.array([0.10, 0.25, 0.50, 1.00, 2.00, 4.00], dtype=float)
LAMBDA_WEIGHTS = np.repeat(1.0 / len(LAMBDAS), len(LAMBDAS))
PSI = np.exp(LAMBDAS) - 1.0 - LAMBDAS

STRICT_INPUTS = False  # set True in the repository once companion surface notebooks have been run

print("Working root:", ROOT.resolve())
print("Output:", OUT.resolve())



## 1. DRVM implementation

The reference upper bound is the one-sided binary-KL bound

$$
m\,\mathrm{kl}(\bar X_m,r_0^U)=\log(1/\gamma).
$$

The deployment monitor uses the manuscript's discrete betting grid and change-point prior

$$
\rho_k=\frac{1}{k(k+1)}.
$$


In [ ]:

def binary_kl(p, q):
    eps = 1e-15
    p = float(np.clip(p, eps, 1-eps))
    q = float(np.clip(q, eps, 1-eps))
    return p*math.log(p/q) + (1-p)*math.log((1-p)/(1-q))


def kl_upper_mean(xbar, m, gamma=GAMMA):
    xbar = float(np.clip(xbar, 0.0, 1.0))
    if xbar >= 1.0:
        return 1.0
    target = math.log(1.0/gamma)
    lo, hi = xbar, 1.0 - 1e-15
    for _ in range(120):
        mid = (lo + hi) / 2.0
        if m * binary_kl(xbar, mid) > target:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2.0


def run_drvm(x, c, alpha=ALPHA):
    x = np.asarray(x, dtype=float)
    if np.any((x < 0) | (x > 1)):
        raise ValueError("DRVM requires X_t in [0,1].")
    if c >= 1.0:
        return {
            "tau": None,
            "status": "not_informatively_monitorable",
            "trajectory": pd.DataFrame({
                "t": np.arange(1, len(x)+1),
                "x": x,
                "C_t": np.ones(len(x)),
            }),
        }

    v = c*(1-c) if c <= 0.5 else 0.25
    A = np.zeros(len(LAMBDAS), dtype=float)
    threshold = 1.0 / alpha
    rows = []
    tau = None

    for idx, xt in enumerate(x):
        t = idx + 1
        rho_t = 1.0 / (t*(t+1.0))
        factor = np.exp(np.clip(LAMBDAS*(xt-c) - PSI*v, -745.0, 700.0))
        A = factor * (A + rho_t)
        C_t = 1.0/(t+1.0) + float(A @ LAMBDA_WEIGHTS)
        rows.append({"t": t, "x": float(xt), "C_t": C_t})
        if tau is None and C_t >= threshold:
            tau = t

    return {
        "tau": tau,
        "status": "signal" if tau is not None else "no_signal",
        "trajectory": pd.DataFrame(rows),
    }


def choose_B_from_calibration(raw_regret):
    r = np.asarray(raw_regret, dtype=float)
    if len(r) == 0 or not np.isfinite(r).all():
        raise ValueError("Calibration regret is empty or non-finite.")
    B = float(np.max(r))
    if B <= 0:
        B = 1.0
    return B


def normalize_regret(raw_regret, B):
    r = np.asarray(raw_regret, dtype=float)
    x = np.minimum(r / float(B), 1.0)
    return x, float(np.mean(r > B))



## 2. Generic surface-to-DRR adapters


In [ ]:

def utility_surface_drr(df, weight_cols, utility_col, time_cols, frozen_weights):
    d = df.copy()
    for c in weight_cols + [utility_col]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=weight_cols + [utility_col])

    keys = d[time_cols].drop_duplicates().sort_values(time_cols)
    oracle = d.groupby(time_cols, as_index=False)[utility_col].max().rename(columns={utility_col:"oracle_value"})

    mask = np.ones(len(d), dtype=bool)
    for c, val in zip(weight_cols, frozen_weights):
        mask &= np.isclose(d[c].to_numpy(float), float(val), rtol=0, atol=1e-8)
    dep = d.loc[mask, time_cols + [utility_col]].copy()
    dep = dep.groupby(time_cols, as_index=False)[utility_col].first().rename(columns={utility_col:"deployed_value"})

    out = oracle.merge(dep, on=time_cols, how="left").sort_values(time_cols).reset_index(drop=True)
    if out["deployed_value"].isna().any():
        raise RuntimeError("Frozen Cartola representation is missing from at least one period in the all-weight surface.")
    out["raw_drr"] = out["oracle_value"] - out["deployed_value"]
    out["raw_drr"] = out["raw_drr"].clip(lower=0)
    return out


def loss_surface_drr(df, theta_hat, loss_col="realized_loss", time_col="date"):
    d = df.copy()
    d["theta"] = pd.to_numeric(d["theta"], errors="coerce")
    d[loss_col] = pd.to_numeric(d[loss_col], errors="coerce")
    d = d.dropna(subset=["theta", loss_col, time_col])

    oracle = d.groupby(time_col, as_index=False)[loss_col].min().rename(columns={loss_col:"oracle_loss"})
    dep = d[np.isclose(d["theta"].to_numpy(float), float(theta_hat), rtol=0, atol=1e-8)]
    dep = dep[[time_col, loss_col]].groupby(time_col, as_index=False)[loss_col].first().rename(columns={loss_col:"deployed_loss"})

    out = oracle.merge(dep, on=time_col, how="left").sort_values(time_col).reset_index(drop=True)
    if out["deployed_loss"].isna().any():
        raise RuntimeError(f"Frozen theta={theta_hat} is missing from at least one day in the all-theta surface.")
    out["raw_drr"] = out["deployed_loss"] - out["oracle_loss"]
    out["raw_drr"] = out["raw_drr"].clip(lower=0)
    return out



## 3. Input discovery and audit

The notebook intentionally refuses to substitute calibration data for the independent reference period. If the clean $D\to C\to R\to E$ inputs are missing, the application is reported as unavailable rather than silently weakening the theorem's conditions.


In [ ]:
def first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

cartola_root = first_existing([
    ROOT / "resultados_fame_temporal_h2_2021_2022_2023_2025_drvm",
    ROOT.parent / "resultados_fame_temporal_h2_2021_2022_2023_2025_drvm",
])

# Legacy H2 root retained only as a fallback for users who already generated it.
cartola_legacy_root = first_existing([
    ROOT / "resultados_fame_temporal_h2_2021_2022_2023_2024",
    ROOT.parent / "resultados_fame_temporal_h2_2021_2022_2023_2024",
])

energy_root = first_existing([
    ROOT / "data" / "fame_energy_temporal_replication_v32_robustness",
    ROOT.parent / "data" / "fame_energy_temporal_replication_v32_robustness",
])

paths = {}

if cartola_root is not None:
    paths.update({
        "cartola_calibration": cartola_root / "03_calibracao_decisao" / "doc_calibration_by_round_all_weights.csv",
        "cartola_frozen": cartola_root / "03_calibracao_decisao" / "frozen_doc_weights.csv",
        "cartola_reference": cartola_root / "08_drvm_inputs" / "posthoc_2024_all_weights_by_round.csv",
        "cartola_deployment": cartola_root / "04_resultados_2025" / "posthoc_2025_all_weights_by_round.csv",
    })

# Fallback only for the first three inputs if the clean V10 files are absent.
if cartola_legacy_root is not None:
    legacy = {
        "cartola_calibration": cartola_legacy_root / "03_calibracao_decisao" / "doc_calibration_by_round_all_weights.csv",
        "cartola_frozen": cartola_legacy_root / "03_calibracao_decisao" / "frozen_doc_weights.csv",
        "cartola_reference": cartola_legacy_root / "04_resultados_2024" / "posthoc_2024_all_weights_by_round.csv",
    }
    for k,p in legacy.items():
        if k not in paths or not paths[k].exists():
            paths[k] = p

if energy_root is not None:
    paths.update({
        "energy_cal_alltheta": energy_root / "drvm_calibration_alltheta_day_level_margin1.csv",
        "energy_test_alltheta": energy_root / "drvm_test_alltheta_day_level_margin1.csv",
        "energy_freezes": energy_root / "theta_freezes_by_margin.csv",
    })

audit_rows = []
for name in [
    "cartola_calibration", "cartola_frozen", "cartola_reference", "cartola_deployment",
    "energy_cal_alltheta", "energy_test_alltheta", "energy_freezes",
]:
    p = paths.get(name)
    audit_rows.append({
        "input": name,
        "path": None if p is None else str(p),
        "exists": bool(p is not None and p.exists()),
    })

input_audit = pd.DataFrame(audit_rows)
input_audit.to_csv(OUT / "input_audit_v10.csv", index=False)
display(input_audit)

if STRICT_INPUTS and not input_audit["exists"].all():
    raise FileNotFoundError("Missing one or more clean D-C-R-E inputs. Run the companion surface notebooks first.")



## 4. Cartola: 2023 freeze, 2024 reference, 2025 monitoring


In [ ]:

cartola_required = ["cartola_calibration", "cartola_frozen", "cartola_reference", "cartola_deployment"]
cartola_ready = all(paths.get(k) is not None and paths[k].exists() for k in cartola_required)

cartola_summary = pd.DataFrame()
cartola_trajectory = pd.DataFrame()

if not cartola_ready:
    print("Cartola clean DRVM inputs are not complete.")
    print("Run FAME_Cartola_H2_extend_to_2025_for_DRVM.ipynb after the original H2 2024 notebook.")
else:
    cal = pd.read_csv(paths["cartola_calibration"])
    ref = pd.read_csv(paths["cartola_reference"])
    dep = pd.read_csv(paths["cartola_deployment"])
    frozen = pd.read_csv(paths["cartola_frozen"]).iloc[0]

    weight_cols = [c for c in ["w_expected", "w_upside", "w_economic"] if c in cal.columns]
    if len(weight_cols) != 3:
        alt = [c for c in ["w_expected", "w_ceiling", "w_economic"] if c in cal.columns]
        if len(alt) == 3:
            weight_cols = alt
        else:
            raise KeyError("Could not identify the three Cartola weight columns.")

    frozen_weights = tuple(float(frozen[c]) for c in weight_cols)
    utility_col = "operational_utility_no_captain"
    time_cols = ["temporada", "rodada_target"]

    drr_cal = utility_surface_drr(cal, weight_cols, utility_col, time_cols, frozen_weights)
    drr_ref = utility_surface_drr(ref, weight_cols, utility_col, time_cols, frozen_weights)
    drr_dep = utility_surface_drr(dep, weight_cols, utility_col, time_cols, frozen_weights)

    B_cartola = choose_B_from_calibration(drr_cal["raw_drr"])
    x_ref, clip_ref = normalize_regret(drr_ref["raw_drr"], B_cartola)
    x_dep, clip_dep = normalize_regret(drr_dep["raw_drr"], B_cartola)

    r0_u = kl_upper_mean(float(np.mean(x_ref)), len(x_ref), GAMMA)
    c = r0_u + DELTA_X
    mon = run_drvm(x_dep, c, ALPHA)

    tr = mon["trajectory"].copy()
    tr = pd.concat([drr_dep.reset_index(drop=True), tr], axis=1)
    tr["threshold"] = 1.0 / ALPHA
    tr.to_csv(OUT / "cartola_drvm_trajectory_v10.csv", index=False)
    cartola_trajectory = tr

    tau = mon["tau"]
    signal_round = None
    if tau is not None:
        signal_round = int(tr.loc[tau-1, "rodada_target"])

    cartola_summary = pd.DataFrame([{
        "application": "Cartola",
        "development": "2021-2022",
        "calibration": 2023,
        "reference": 2024,
        "deployment": 2025,
        "frozen_weights": str(frozen_weights),
        "B_from_calibration": B_cartola,
        "n_reference": len(x_ref),
        "n_deployment": len(x_dep),
        "mean_raw_drr_reference": float(drr_ref["raw_drr"].mean()),
        "mean_raw_drr_deployment": float(drr_dep["raw_drr"].mean()),
        "mean_x_reference": float(np.mean(x_ref)),
        "mean_x_deployment": float(np.mean(x_dep)),
        "r0_upper": r0_u,
        "c": c,
        "delta_x": DELTA_X,
        "reference_clip_fraction": clip_ref,
        "deployment_clip_fraction": clip_dep,
        "status": mon["status"],
        "tau_index": tau,
        "signal_round_2025": signal_round,
        "max_C_t": float(tr["C_t"].max()) if len(tr) else np.nan,
        "final_C_t": float(tr["C_t"].iloc[-1]) if len(tr) else np.nan,
    }])
    cartola_summary.to_csv(OUT / "cartola_drvm_summary_v10.csv", index=False)
    display(cartola_summary)



## 5. Energy: eight clean annual monitoring episodes

For deployment year $e$, the representation frozen from the previous standard replication is held fixed across the independent reference year $e-1$ and deployment year $e$. The predictor may continue to evolve according to the original expanding-window protocol; DRR always compares candidate representations under the same period-specific predictive input.

There is no known "true" change point in real data, so the reported quantity is **time to first signal from deployment start**, not detection delay.


In [ ]:

energy_required = ["energy_cal_alltheta", "energy_test_alltheta", "energy_freezes"]
energy_ready = all(paths.get(k) is not None and paths[k].exists() for k in energy_required)

energy_summary = pd.DataFrame()
energy_trajectories = []

if not energy_ready:
    print("Energy clean DRVM inputs are not complete.")
    print("Run FAME_Energy_export_alltheta_surfaces_for_DRVM.ipynb first.")
else:
    cal_all = pd.read_csv(paths["energy_cal_alltheta"])
    test_all = pd.read_csv(paths["energy_test_alltheta"])
    freezes = pd.read_csv(paths["energy_freezes"])

    cal_all["date"] = pd.to_datetime(cal_all["date"])
    test_all["date"] = pd.to_datetime(test_all["date"])

    rows = []
    for year in range(2018, 2026):
        prev_rid = f"E{year-1}"
        dep_rid = f"E{year}"

        for model in ["ENTSOE", "SARIMAX", "LSTM"]:
            fr = freezes[
                freezes["replication_id"].eq(prev_rid)
                & np.isclose(freezes["margin"].astype(float), 1.0)
            ]
            if fr.empty:
                print("skip missing freeze", prev_rid, model)
                continue
            theta_hat = float(fr.iloc[0][f"theta_{model}"])

            cal_df = cal_all[
                cal_all["replication_id"].eq(prev_rid)
                & cal_all["model"].eq(model)
            ].copy()
            ref_df = test_all[
                test_all["replication_id"].eq(prev_rid)
                & test_all["model"].eq(model)
            ].copy()
            dep_df = test_all[
                test_all["replication_id"].eq(dep_rid)
                & test_all["model"].eq(model)
            ].copy()

            if cal_df.empty or ref_df.empty or dep_df.empty:
                print("skip incomplete surfaces", year, model)
                continue

            drr_cal = loss_surface_drr(cal_df, theta_hat)
            drr_ref = loss_surface_drr(ref_df, theta_hat)
            drr_dep = loss_surface_drr(dep_df, theta_hat)

            B = choose_B_from_calibration(drr_cal["raw_drr"])
            x_ref, clip_ref = normalize_regret(drr_ref["raw_drr"], B)
            x_dep, clip_dep = normalize_regret(drr_dep["raw_drr"], B)

            r0_u = kl_upper_mean(float(np.mean(x_ref)), len(x_ref), GAMMA)
            c = r0_u + DELTA_X
            mon = run_drvm(x_dep, c, ALPHA)

            tr = mon["trajectory"].copy()
            tr = pd.concat([drr_dep.reset_index(drop=True), tr], axis=1)
            tr["deployment_year"] = year
            tr["model"] = model
            tr["theta_frozen_from_previous_replication"] = theta_hat
            tr["threshold"] = 1.0 / ALPHA
            energy_trajectories.append(tr)

            tau = mon["tau"]
            signal_date = None
            if tau is not None:
                signal_date = tr.loc[tau-1, "date"]

            rows.append({
                "application": "Energy",
                "deployment_year": year,
                "reference_replication": prev_rid,
                "deployment_replication": dep_rid,
                "model": model,
                "margin": 1.0,
                "theta_hat": theta_hat,
                "B_from_calibration": B,
                "n_reference": len(x_ref),
                "n_deployment": len(x_dep),
                "mean_raw_drr_reference": float(drr_ref["raw_drr"].mean()),
                "mean_raw_drr_deployment": float(drr_dep["raw_drr"].mean()),
                "mean_x_reference": float(np.mean(x_ref)),
                "mean_x_deployment": float(np.mean(x_dep)),
                "r0_upper": r0_u,
                "c": c,
                "delta_x": DELTA_X,
                "reference_clip_fraction": clip_ref,
                "deployment_clip_fraction": clip_dep,
                "status": mon["status"],
                "time_to_signal_days": tau,
                "signal_date": signal_date,
                "max_C_t": float(tr["C_t"].max()) if len(tr) else np.nan,
                "final_C_t": float(tr["C_t"].iloc[-1]) if len(tr) else np.nan,
            })

    energy_summary = pd.DataFrame(rows)
    energy_summary.to_csv(OUT / "energy_drvm_summary_v10.csv", index=False)
    if energy_trajectories:
        energy_trajectory = pd.concat(energy_trajectories, ignore_index=True)
        energy_trajectory.to_csv(OUT / "energy_drvm_trajectories_v10.csv", index=False)
    display(energy_summary)



## 6. Application-level figures

Figures are descriptive deployment traces. A signal is evidence against operational validity of the frozen representation; it does **not** by itself establish that recalibration is the optimal intervention.


In [ ]:

if len(cartola_trajectory):
    plt.figure(figsize=(8.5, 4.8))
    plt.plot(cartola_trajectory["rodada_target"], cartola_trajectory["C_t"], marker="o")
    plt.axhline(1.0/ALPHA, linestyle="--")
    plt.xlabel("2025 round")
    plt.ylabel("DRVM evidence $C_t$")
    plt.yscale("log")
    plt.tight_layout()
    plt.savefig(OUT / "fig_cartola_drvm_2025.pdf")
    plt.savefig(OUT / "fig_cartola_drvm_2025.png", dpi=220)
    plt.close()

if len(energy_summary):
    focus = energy_summary[
        energy_summary["deployment_year"].eq(2021)
        & energy_summary["model"].eq("ENTSOE")
    ]
    if not focus.empty and energy_trajectories:
        all_tr = pd.concat(energy_trajectories, ignore_index=True)
        tr = all_tr[
            all_tr["deployment_year"].eq(2021)
            & all_tr["model"].eq("ENTSOE")
        ].copy()
        if len(tr):
            plt.figure(figsize=(8.5, 4.8))
            plt.plot(pd.to_datetime(tr["date"]), tr["C_t"])
            plt.axhline(1.0/ALPHA, linestyle="--")
            plt.xlabel("Deployment date")
            plt.ylabel("DRVM evidence $C_t$")
            plt.tight_layout()
            plt.savefig(OUT / "fig_energy_drvm_ENTSOE_E2021.pdf")
            plt.savefig(OUT / "fig_energy_drvm_ENTSOE_E2021.png", dpi=220)
            plt.close()



## 7. Compact manuscript-ready summary and archive


In [ ]:

summary_parts = []
if len(cartola_summary):
    summary_parts.append(cartola_summary.assign(domain="Cartola"))
if len(energy_summary):
    # Keep the full Energy table separate; this compact table reports signal counts by source.
    energy_compact = (
        energy_summary.groupby("model", as_index=False)
        .agg(
            n_episodes=("deployment_year", "size"),
            n_signals=("status", lambda s: int((s == "signal").sum())),
            n_not_informative=("status", lambda s: int((s == "not_informatively_monitorable").sum())),
            median_time_to_signal=("time_to_signal_days", "median"),
            mean_reference_drr=("mean_raw_drr_reference", "mean"),
            mean_deployment_drr=("mean_raw_drr_deployment", "mean"),
        )
    )
    energy_compact.to_csv(OUT / "energy_drvm_compact_v10.csv", index=False)
    display(energy_compact)

zip_path = shutil.make_archive(str(ROOT / "results_drvm_real_v10"), "zip", root_dir=OUT)
print("Archive:", zip_path)
print("Important: if the input audit contains missing files, this archive contains only the audit and any completed domain results.")
